# 4-4 다중 분류 출력 심화

강의 원문 대신 직접 작성하고 실행한 코드와 학습 메모를 정리했습니다.


In [1]:
# 검증 가능 정답 코드
import torch
logits = torch.tensor([[2., 1., 0.], [0., 1., 2.]])
# dim=0은 sample 사이를 정규화하는 잘못된 경로이고, dim=1은 각 sample의 class 축을 정규화합니다.
wrong = torch.softmax(logits, dim=0)
correct = torch.softmax(logits, dim=1)
# 각 행 확률 합이 1인지 확인하면 class 축 선택 계약을 출력값으로 감사할 수 있습니다.
print("wrong_row_sums:", [round(v, 1) for v in wrong.sum(dim=1).tolist()])
print("correct_row_sums:", [round(v, 1) for v in correct.sum(dim=1).tolist()])
print("selected_dim:", 1)

wrong_row_sums: [1.5, 1.5]
correct_row_sums: [1.0, 1.0]
selected_dim: 1


In [2]:
# 검증 가능 정답 코드
import torch
from torch import nn

def multiclass_result(logits, target):
    assert logits.ndim == 2 and target.shape == (logits.shape[0],)
    assert target.dtype == torch.long
    assert int(target.min()) >= 0 and int(target.max()) < logits.shape[1]
    loss = nn.CrossEntropyLoss()(logits, target)  # Softmax 전 raw 점수를 전달합니다. ## 객체를 만드는 클래스.Loss()뒤에 넣기
        # 확률과 class 예측은 해석용으로 loss 계산 뒤 별도 생성해 CrossEntropy 입력을 바꾸지 않습니다.
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(logits, dim=-1)
    return loss, probs, preds

logits = torch.tensor([[2., 1., 0.], [0., 1., 2.]])
target = torch.tensor([0, 2], dtype=torch.long)
loss, probs, preds = multiclass_result(logits, target)
print("loss_scalar:", loss.ndim == 0)
print("row_sums:", [round(v, 1) for v in probs.sum(-1).tolist()])
print("preds:", preds.tolist())

loss_scalar: True
row_sums: [1.0, 1.0]
preds: [0, 2]


In [3]:
# 검증 가능 정답 코드
runs = {
    "A": {"accuracy": 0.88, "row_sum_error": 0.50, "target_dtype": "float", "safety_recall": 0.81},
    "B": {"accuracy": 0.86, "row_sum_error": 0.00, "target_dtype": "long", "safety_recall": 0.78},
    "C": {"accuracy": 0.90, "row_sum_error": 0.00, "target_dtype": "long", "safety_recall": None},
}
approved, remeasure, rejected = [], [], {}
# 품질 점수를 보기 전에 확률 행 합과 target dtype이라는 Tensor 계약을 먼저 검사합니다.
for name, run in runs.items():
    tensor_ok = run["row_sum_error"] <= 1e-6 and run["target_dtype"] == "long"
    if not tensor_ok:
        rejected[name] = "tensor_contract"
    # 안전 recall이 누락된 C는 성능을 추정하지 않고 재측정 상태로 보존합니다.
    elif run["safety_recall"] is None:
        remeasure.append(name)
    elif run["accuracy"] >= 0.85 and run["safety_recall"] >= 0.75:
        approved.append(name)
    else:
        rejected[name] = "quality"
# 승인 목록이 만들어진 뒤에만 같은 계약의 accuracy를 비교해 최종 후보를 정합니다.
selected = max(approved, key=lambda name: runs[name]["accuracy"]) if approved else "보류"
print("approved:", approved)
print("remeasure:", remeasure)
print("rejected:", rejected)
print("selected:", selected)

approved: ['B']
remeasure: ['C']
rejected: {'A': 'tensor_contract'}
selected: B
